<a href="https://colab.research.google.com/github/codebysumit/cryptography-algorithms/blob/master/notebooks/hill_cipher.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hill Cipher

## History
The Hill Cipher was invented by Lester S. Hill in 1929. It was one of the first ciphers to bring real linear algebra into cryptography. Instead of shifting or substituting one character at a time like Caesar, Affine, or Vigenere, Hill Cipher encrypts several characters together as a block, using matrix multiplication. This was a big step forward at the time, because it hides letter frequency much better than single character ciphers.

## What is Hill Cipher?
Hill Cipher treats a group of letters as a vector of numbers, and multiplies that vector by a square matrix, called the **key matrix**. The whole cipher works using **modular arithmetic on 26**, because it operates on the English alphabet A to Z.

A = 0, B = 1, C = 2, ... Z = 25

The plaintext is split into blocks. If the key matrix is size $n \times n$, then each block has exactly $n$ letters. Each block is converted into a column vector of numbers, multiplied by the key matrix, and reduced modulo 26 to get the ciphertext block.

This notebook only works on alphabet letters (A-Z). Spaces, punctuation, and numbers are removed before encryption, because the matrix math only makes sense on a clean block of letters. This is different from the earlier ciphers in this course, which worked directly on the full printable ASCII range.

## Cryptography Algorithm

### Constants and Symbols
*   $N = 26$ (size of the English alphabet)
*   $K$ = the key matrix, size $n \times n$
*   $n$ = block size, same as the size of the key matrix
*   $P$ = plaintext block, written as a column vector of $n$ numbers
*   $C$ = ciphertext block, written as a column vector of $n$ numbers
*   $K^{-1}$ = the modular inverse of the key matrix, used only for decryption

### 1. Encryption
Each plaintext block is multiplied by the key matrix and reduced modulo 26:

$$C = (K \cdot P) \bmod 26$$

### 2. Decryption
Decryption needs the **inverse of the key matrix**, not the key matrix itself:

$$P = (K^{-1} \cdot C) \bmod 26$$

Finding $K^{-1}$ is the hardest part of Hill Cipher, so let's break it down step by step.

### 3. Step by Step: Finding the Inverse of the Key Matrix

**Step 1: Find the determinant of $K$, then reduce it modulo 26.**

$$d = \det(K) \bmod 26$$

**Step 2: Find the modular inverse of $d$ under mod 26.**

We need a number $d^{-1}$ such that:

$$(d \cdot d^{-1}) \bmod 26 = 1$$

This inverse only exists if $\gcd(d, 26) = 1$. This is why the key matrix cannot be chosen randomly, its determinant must be coprime with 26, meaning it cannot be even, and it cannot be a multiple of 13.

**Step 3: Find the matrix of cofactors of $K$.**

For every cell $(i, j)$ in $K$, remove row $i$ and column $j$ to get a smaller matrix called the **minor**. The cofactor at that position is:

$$cof_{i,j} = (-1)^{i+j} \cdot \det(\text{minor}_{i,j})$$

**Step 4: Transpose the cofactor matrix to get the adjugate.**

$$adj(K) = cof(K)^T$$

**Step 5: Multiply the adjugate by $d^{-1}$, and reduce modulo 26.**

$$K^{-1} = (d^{-1} \cdot adj(K)) \bmod 26$$

If every step above was done correctly, multiplying $K$ by $K^{-1}$ modulo 26 always gives back the identity matrix.

$$(K \cdot K^{-1}) \bmod 26 = I$$

### 4. Fully Worked Example (By Hand, 2x2)

Let's encrypt the word **HI** using the key matrix:

$$K = \begin{bmatrix} 3 & 3 \\ 2 & 7 \end{bmatrix}$$

**Step 1: Convert letters to numbers.**
H = 7, I = 8, so the plaintext vector is:

$$P = \begin{bmatrix} 7 \\ 8 \end{bmatrix}$$

**Step 2: Multiply the key matrix by the plaintext vector.**

$$K \cdot P = \begin{bmatrix} 3 \times 7 + 3 \times 8 \\ 2 \times 7 + 7 \times 8 \end{bmatrix} = \begin{bmatrix} 21 + 24 \\ 14 + 56 \end{bmatrix} = \begin{bmatrix} 45 \\ 70 \end{bmatrix}$$

**Step 3: Reduce modulo 26.**

$$45 \bmod 26 = 19 \qquad 70 \bmod 26 = 18$$

So the ciphertext vector is $\begin{bmatrix} 19 \\ 18 \end{bmatrix}$. Converting numbers back to letters: 19 = T, 18 = S.

**Ciphertext = TS**

**Now let's decrypt TS back to HI.**

**Step 1: Find the determinant of $K$.**

$$\det(K) = (3 \times 7) - (3 \times 2) = 21 - 6 = 15$$

$$d = 15 \bmod 26 = 15$$

**Step 2: Find the modular inverse of 15 under mod 26.**

We are looking for a number $x$ such that $(15 \times x) \bmod 26 = 1$. Testing values, $15 \times 7 = 105$, and $105 \bmod 26 = 1$. So:

$$d^{-1} = 7$$

**Step 3: Find the cofactor matrix.** For a 2x2 matrix, this step has a shortcut: swap the two diagonal values, and negate the two off diagonal values.

$$cof(K) = \begin{bmatrix} 7 & -2 \\ -3 & 3 \end{bmatrix}$$

**Step 4: Transpose it to get the adjugate.** For a 2x2 matrix the transpose of this particular cofactor pattern gives:

$$adj(K) = \begin{bmatrix} 7 & -3 \\ -2 & 3 \end{bmatrix}$$

**Step 5: Multiply by $d^{-1} = 7$, then reduce mod 26.**

$$K^{-1} = \left( 7 \times \begin{bmatrix} 7 & -3 \\ -2 & 3 \end{bmatrix} \right) \bmod 26 = \begin{bmatrix} 49 & -21 \\ -14 & 21 \end{bmatrix} \bmod 26 = \begin{bmatrix} 23 & 5 \\ 12 & 21 \end{bmatrix}$$

**Step 6: Multiply $K^{-1}$ by the ciphertext vector.**

$$K^{-1} \cdot C = \begin{bmatrix} 23 \times 19 + 5 \times 18 \\ 12 \times 19 + 21 \times 18 \end{bmatrix} = \begin{bmatrix} 437 + 90 \\ 228 + 378 \end{bmatrix} = \begin{bmatrix} 527 \\ 606 \end{bmatrix}$$

**Step 7: Reduce modulo 26.**

$$527 \bmod 26 = 7 \qquad 606 \bmod 26 = 8$$

7 = H, 8 = I, so we correctly get back **HI**. This confirms the whole cycle: $K$ encrypts, $K^{-1}$ decrypts, and everything checks out.

### Key Requirements
*   The key matrix must be a **square matrix** ($n \times n$).
*   The **determinant of the key matrix, mod 26, must be coprime with 26**. If it is not, the matrix has no inverse mod 26, and decryption becomes impossible.
*   A larger key matrix (like 3x3 or 4x4) hides letter patterns even better than a 2x2, because more letters get mixed together in every block.

### 1. Import Dependencies

In [1]:
import random
import math
import numpy as np

### 2. Helper Utilities (Letters, Cleaning, Padding)

In [2]:
def clean_text(text: str) -> str:
    # Hill Cipher math only works on plain A-Z letters, so we strip everything else
    return "".join(ch for ch in text.upper() if ch.isalpha())

def pad_text(text: str, block_size: int) -> str:
    # pad with 'X' so the text splits evenly into blocks of block_size letters
    while len(text) % block_size != 0:
        text += "X"
    return text

### 3. Modular Arithmetic Utilities (Determinant, Inverse, Adjugate)

In [3]:
def mod_inverse(a: int, m: int) -> int:
    # find x such that (a * x) % m == 1
    a = a % m
    for x in range(1, m):
        if (a * x) % m == 1:
            return x
    return None  # no inverse exists if a and m are not coprime

def matrix_determinant_mod(matrix: np.ndarray, mod: int) -> int:
    # numpy gives a floating point determinant, so we round it before taking mod
    det = int(round(np.linalg.det(matrix)))
    return det % mod

def get_minor(matrix: np.ndarray, row: int, col: int) -> np.ndarray:
    # remove the given row and column to get the minor matrix
    return np.delete(np.delete(matrix, row, axis=0), col, axis=1)

def cofactor_matrix(matrix: np.ndarray) -> np.ndarray:
    n = matrix.shape[0]
    cof = np.zeros((n, n), dtype=int)
    for i in range(n):
        for j in range(n):
            minor = get_minor(matrix, i, j)
            minor_det = int(round(np.linalg.det(minor))) if n > 1 else 1
            cof[i, j] = ((-1) ** (i + j)) * minor_det
    return cof

def matrix_mod_inverse(matrix: np.ndarray, mod: int) -> np.ndarray:
    det = matrix_determinant_mod(matrix, mod)
    det_inv = mod_inverse(det, mod)

    if det_inv is None:
        raise ValueError(f"Determinant {det} has no inverse mod {mod}. Choose a different key matrix.")

    cof = cofactor_matrix(matrix)
    adjugate = cof.T
    inverse = (det_inv * adjugate) % mod
    return inverse

def is_invertible_mod(matrix: np.ndarray, mod: int) -> bool:
    det = matrix_determinant_mod(matrix, mod)
    return math.gcd(det, mod) == 1

### 4. Generate a Random Valid Key Matrix

In [4]:
def generate_random_key(n: int, mod: int = 26) -> np.ndarray:
    # keep generating random n x n matrices until we find one that is invertible mod 26
    while True:
        matrix = np.array([[random.randint(0, mod - 1) for _ in range(n)] for _ in range(n)])
        if is_invertible_mod(matrix, mod):
            return matrix

### 5. Encryption

In [5]:
def encrypt(text: str, key_matrix: np.ndarray, mod: int = 26) -> str:
    n = key_matrix.shape[0]

    if not is_invertible_mod(key_matrix, mod):
        raise ValueError("This key matrix has no inverse mod 26, so it cannot be used for Hill Cipher.")

    clean = clean_text(text)
    padded = pad_text(clean, n)

    cipher_text = ""
    for i in range(0, len(padded), n):
        block = padded[i:i + n]
        plain_vector = np.array([ord(ch) - ord('A') for ch in block])

        cipher_vector = (key_matrix.dot(plain_vector)) % mod

        cipher_text += "".join(chr(val + ord('A')) for val in cipher_vector)

    return cipher_text

### 6. Decryption

In [6]:
def decrypt(cipher_text: str, key_matrix: np.ndarray, mod: int = 26) -> str:
    n = key_matrix.shape[0]
    inverse_key = matrix_mod_inverse(key_matrix, mod)

    plain_text = ""
    for i in range(0, len(cipher_text), n):
        block = cipher_text[i:i + n]
        cipher_vector = np.array([ord(ch) - ord('A') for ch in block])

        plain_vector = (inverse_key.dot(cipher_vector)) % mod

        plain_text += "".join(chr(val + ord('A')) for val in plain_vector)

    return plain_text

### 7. Verify the Hand Worked Example in Code

In [7]:
hand_key = np.array([
    [3, 3],
    [2, 7]
])

hand_plaintext = "HI"
hand_cipher = encrypt(hand_plaintext, hand_key)
hand_decrypted = decrypt(hand_cipher, hand_key)

print(f"Key Matrix:\n{hand_key}")
print(f"Plaintext: {hand_plaintext}")
print(f"Encrypted: {hand_cipher}  (should match TS from the hand worked example)")
print(f"Decrypted: {hand_decrypted}")

Key Matrix:
[[3 3]
 [2 7]]
Plaintext: HI
Encrypted: TS  (should match TS from the hand worked example)
Decrypted: HI


### 8. Example usage (Random Key, Longer Message)

In [8]:
key = generate_random_key(n=3)
print(f"Generated Random Key Matrix:\n{key}")

Generated Random Key Matrix:
[[15  9 19]
 [ 8 17 14]
 [19 16  8]]


In [10]:
plaintext = "TOP secret Massage! Agent PI, visit Area fifty-one"
print(f"Original Plain Text: {plaintext}")
print(f"Cleaned Text (letters only): {clean_text(plaintext)}")

cipher_text = encrypt(plaintext, key)
print("Encrypted:", cipher_text)

decrypted_text = decrypt(cipher_text, key)
print("Decrypted:", decrypted_text)

match = pad_text(clean_text(plaintext), key.shape[0]) == decrypted_text
print(f"Verification Match:{match}")

Original Plain Text: TOP secret Massage! Agent PI, visit Area fifty-one
Cleaned Text (letters only): TOPSECRETMASSAGEAGENTPIVISITAREAFIFTYONE
Encrypted: UCDGGGCCTCKIUUASMUSZUUEJSOKKADZYMGZUFOECRE
Decrypted: TOPSECRETMASSAGEAGENTPIVISITAREAFIFTYONEXX
Verification Match:True
